# Build treebanks from XML corpora

Treebanks being parsed are. Only excluded text is Ogimaans: 
1. Gii-nitaa-aadisooke
2. Akawe Niwii-tibaajim
3. Chi-mewinzha
4. Living Our Language
5. Aanjikiing
6. Anooj inaajimod
7. Plums or Nuts
8. The Dogs' Children
9. Awesiinyensag
10. Mino-doodaading
11. Naadamaading
12. Wiijikiiwending
13. Portage Lake
14. Oshkaabewis
15. OPD Sentences
16. Dibaajimowinan

IMPORTANT: Rerunning this currently just appends to files of the same name. Solutions are either:
1) change logic so that it rewrites the existing file
2) when reparsing change the file name/delete the existing file of the same name

In [1]:
from grammar_modules.disambiguation import REPO_ROOT
from treebank_modules.corpus import build_treebank_from_xml_sentences

import re
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

In [2]:
import string

def single_line(text):
    return re.sub(r"\s+", " ", text).strip()

def is_punct(form):
    return all(c in string.punctuation for c in form)

def has_analysis(token):
    for fst_output in token.findall("fst_output"):
        analysis = (fst_output.get("analysis") or "")
        if analysis: 
            return True
    return False

def is_fully_parsed_sent(sent):
    tokens = sent.findall(".//token")
    if not tokens:
        return False

    return all(is_punct(token.get("word")) or has_analysis(token) 
               for token in tokens)
        
def parse_xml_sentences(xml_path, print_sentences=True):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    sentences = []

    for paragraph in root.findall(".//paragraph"):
        current_para = paragraph.get("num")

        eng_node = paragraph.find("text_eng")
        eng_para = eng_node.text if eng_node is not None else None

        if eng_para is not None:
            eng_sentences = re.split(r"(?<=[.!?])\s*", eng_para)
            eng_sentences = [s for s in eng_sentences if s]
        else:
            eng_sentences = []

        oj_sentences = paragraph.findall("text_ojb/sentence")

        for sent_i, sent in enumerate(oj_sentences):
            
            if not is_fully_parsed_sent(sent):
                continue
            
            sent_text_node = sent.find("sent_text")
            oj_sent_text = (
                single_line(sent_text_node.text)
                if sent_text_node is not None and sent_text_node.text is not None
                else None
            )

            if oj_sent_text is None:
                continue

            current_sent = sent.get("num")

            if len(eng_sentences) > sent_i:
                en_sent_text = single_line(eng_sentences[sent_i])
            else:
                en_sent_text = "No English translation."

            sentences.append((oj_sent_text, en_sent_text))

            if print_sentences:
                print(
                    "Para ", current_para,
                    "Sent ", current_sent,
                    "Oj sentence:", oj_sent_text,
                    "En translation:", en_sent_text,
                )
                print("-" * 40)

    return sentences

In [3]:
XML_ROOT = REPO_ROOT / "data" / "corpus"
OSHKAABEWIS_ROOT = XML_ROOT / "Oshkaabewis"
TREEBANK_ROOT = REPO_ROOT / "data" / "treebanks" / "AC"

In [4]:
for file in XML_ROOT.iterdir():
    if ".DS_Store" not in file.name and "Oshkaabewis" not in file.name:
        sents = parse_xml_sentences(file, False)
        path = TREEBANK_ROOT / (file.stem + ".conllu")
        build_treebank_from_xml_sentences(
            sentences=sents,
            corpus_path=path,
            batch_size=500
        )

FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/OjibweMorph-v1_2_1.fomabin
Processed batch 1: 209 sentences added.
Added 209 sentences to 2026-10-07_Plums-or-nuts.conllu.
FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/OjibweMorph-v1_2_1.fomabin
Processed batch 1: 500 sentences added.
Processed batch 2: 1000 sentences added.
Processed batch 3: 1407 sentences added.
Added 1407 sentences to 2026-10-07_Akawe_Niwii-tibaajim.conllu.
FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/OjibweMorph-v1_2_1.fomabin
Processed batch 1: 500 sentences added.
Processed batch 2: 1000 sentences added.
Processed batch 3: 1165 sentences added.
Added 1165 sentences to 2026-10-07_Portage_Lake.conllu.
FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/OjibweMorph-v1_2_1.fomabin
Processed batch 1: 337 sentences added.
Added 337 sentences to 2026-10-0

In [5]:
sents = []
for file in OSHKAABEWIS_ROOT.iterdir():
    if ".DS_Store" not in file.name:
        sents.extend(parse_xml_sentences(file, False))
        
path = TREEBANK_ROOT / ("20260-10-07_Oshkaabewis.conllu")
build_treebank_from_xml_sentences(
    sentences=sents,
    corpus_path=path,
    batch_size=500
)

FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/OjibweMorph-v1_2_1.fomabin
Processed batch 1: 500 sentences added.
Processed batch 2: 1000 sentences added.
Processed batch 3: 1500 sentences added.
Processed batch 4: 2000 sentences added.
Processed batch 5: 2500 sentences added.
Processed batch 6: 3000 sentences added.
Processed batch 7: 3500 sentences added.
Processed batch 8: 3715 sentences added.
Added 3715 sentences to 20260-10-07_Oshkaabewis.conllu.


3715